# Publication snapshot v0.4.0
整理自既有研究；已移除outputs、憑證與私人路徑。見README與docs/limitations.md。
保留歷史實驗設計供檢視，不代表已修正資料切分或已在乾淨環境重跑。


# 蒸餾 Phase 1 — 產生「老師標籤」訓練資料

目標：做出一份可拿去訓練學生小模型的資料集。流程：

1. 用 **ollama** 的幾個小模型，對文章各生一份摘要（製造對/錯混合的學生訓練輸入）。
2. 用 **GPT-4o v2（中性 prompt）當老師**，替每個 (文章, 摘要) 標上 `teacher_label`（0=一致 / 1=幻覺）與完整錯誤報告。
3. 存成 `distill_data.jsonl`（逐行寫，防截斷）+ 一份精簡 `distill_data.csv`。

> 重要：**FaithBench 不要拿來訓練**（它是你的人工金標評估集）。這裡的訓練資料用你自己的文章（CNNDM 等）+ ollama 生摘要 + GPT-4o 銀標籤。
> 沒有 ollama / 沒有 key 也能跑：會用 MOCK 把整條管線跑完驗證結構（但 MOCK 標籤不可當真資料）。
> 訓練（Phase 2）是另一本 notebook，需要 GPU。


## 0. 設定

In [ ]:
import os, re, json, urllib.request

ARTICLES_CSV = os.environ.get("ARTICLES_CSV", "data/articles.csv")
ARTICLE_COL   = "article"
MAX_ARTICLES  = 1000            # 先小；穩了再放大到幾百

OLLAMA_MODELS = ["qwen3:4b", "llama3.2:3b"]   # 生摘要用；改成你 `ollama list` 有的
OLLAMA_URL    = "http://localhost:11434"

TEACHER_MODEL = "gpt-4o"      # 老師；需環境變數 OPENAI_API_KEY

OUT_JSONL = "distill_data.jsonl"
OUT_CSV   = "distill_data.csv"

import pandas as pd
print("設定完成。")

## 1. 文章來源

In [ ]:
# SAMPLE_ARTICLES = [
#     "Apollo 11 launched on July 16, 1969. Four days later, Neil Armstrong became the first person to walk on the Moon, followed by Buzz Aldrin. Michael Collins remained in lunar orbit aboard the command module.",
#     "The company reported revenue of $2 billion in 2023, up 10% from the previous year. Net profit rose to $300 million. The board approved a dividend of $0.50 per share.",
#     "Mount Everest, at 8,849 metres, is the highest mountain above sea level. It lies on the border between Nepal and China. Hundreds of climbers attempt the summit each spring.",
#     "Marie Curie won the Nobel Prize in Physics in 1903, shared with Pierre Curie and Henri Becquerel. In 1911 she won the Nobel Prize in Chemistry, becoming the first person to win Nobels in two sciences.",
#     "The city council approved a new bike-lane network on Tuesday. The project adds 25 kilometres of protected lanes over two years at a cost of $40 million.",
# ]
if ARTICLES_CSV:
    dfa = pd.read_csv(ARTICLES_CSV)
    articles = dfa[ARTICLE_COL].dropna().astype(str).tolist()
    print(f"從 {ARTICLES_CSV} 載入 {len(articles)} 篇")
else:
    articles = SAMPLE_ARTICLES
    print(f"使用內建 {len(articles)} 篇範例")
articles = articles[:MAX_ARTICLES]
print("本次文章數：", len(articles))

## 2. 用 ollama 生摘要（學生的訓練輸入）

In [ ]:
def ollama_available():
    try:
        with urllib.request.urlopen(OLLAMA_URL + "/api/tags", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

SUMMARIZE_PROMPT = "Summarize the following news article in 2-3 sentences. Output only the summary.\n\nARTICLE:\n{article}\n\nSUMMARY:"
def gen_summary_ollama(article, model):
    import ollama
    r = ollama.chat(model=model, stream=False,
                    messages=[{"role":"user","content":SUMMARIZE_PROMPT.format(article=article)}])
    out = r["message"]["content"]
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S|re.I).strip()

OLLAMA_OK = ollama_available()
if not OLLAMA_OK:
    print("⚠️ 未連到 ollama → 生摘要用 MOCK（取文章前兩句），僅供管線測試。")
def gen_summary(article, model):
    if OLLAMA_OK:
        try:
            return gen_summary_ollama(article, model)
        except Exception as e:
            return f"[gen failed: {e}]"
    sents = [p for p in re.split(r"(?<=[.!?])\s+", article.strip()) if p]
    return " ".join(sents[:2])
print("摘要產生器就緒。OLLAMA_OK =", OLLAMA_OK)

## 3. 用 GPT-4o v2 當老師標籤（中性 / Benign-aware prompt）

In [ ]:
# ============================================================
# 修正：老師 prompt 之前含有未跳脫的大括號 {"sentence_index":...}，
# 導致 .format() 報 KeyError('"sentence_index"') → teacher_label 全 -1。
# 這格用「沒有大括號」的條列版覆寫 detect_errors_prompt。
# 放在「## 3 老師標籤」那格之後、「## 4 執行」之前，從這格往下重跑即可。
# ============================================================

detect_errors_prompt = """
You will be given a news ARTICLE (the ONLY ground truth) and a SUMMARY of it.

INPUT: the SUMMARY is a JSON object with a key "sentences" (a list of strings, one sentence each).
The index of each sentence (0, 1, 2, ...) is its sentence_index.

DO NOT assume the summary is flawed. It may be fully faithful, or it may contain errors. Judge faithfulness.

Count a sentence as a factual error ONLY IF it:
- directly CONTRADICTS the article, OR
- asserts a specific fact (named entity, number/date/amount, event, or who-did-what-to-whom)
  that the article does NOT support.
DO NOT flag: a paraphrase, compression, or entailed inference; correct world knowledge that does not
contradict the article; or wording/style differences. When unsure, prefer NOT to flag.

ERROR TYPES (choose the single most important one per flagged sentence):
Wrong Entity, Wrong Number, Fabricated Fact, Negation Flip, Role Inversion.

OUTPUT STRICT JSON only, with exactly these keys:
- "result": "Has Error" if at least one sentence is a genuine factual error, otherwise "No Error".
- "errors": a list of objects; each object has the keys sentence_index (integer),
  error_type (one of the five types), explanation (string), and proposed_fix_sentence (string).
  Use an empty list when there are no errors.

Output nothing outside the JSON object.

ARTICLE:
{article}

SUMMARY JSON:
{flawed_summary_json}

Now judge the summary and return the JSON object.
"""
print("已套用修正版老師 prompt（無大括號，.format 不會再炸）。重跑第 4、5 格。")
_openai_client = None
def _get_openai():
    global _openai_client
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    return _openai_client
def split_sentences(text):
    return [p for p in re.split(r"(?<=[.!?])\s+", str(text).strip()) if p]
def teacher_label(article, summary):
    client = _get_openai()
    prompt = detect_errors_prompt.format(
        article=article,
        flawed_summary_json=json.dumps({"sentences": split_sentences(summary)}, ensure_ascii=False))
    resp = client.chat.completions.create(
        model=TEACHER_MODEL, temperature=0, response_format={"type": "json_object"},
        messages=[{"role":"system","content":"You are a precise factual error detector."},
                  {"role":"user","content":prompt}])
    rep = json.loads(resp.choices[0].message.content)
    return (1 if rep.get("result") == "Has Error" else 0), rep

TEACHER_OK = bool(os.environ.get("OPENAI_API_KEY"))
if not TEACHER_OK:
    print("⚠️ 未設 OPENAI_API_KEY → 老師標籤用 MOCK（數字啟發式），僅供管線測試，不可當真資料。")
def _nums(s):
    out = set()
    for m in re.findall(r"\d[\d,\.]*", str(s)):
        t = m.strip(",.").replace(",", "")
        if t: out.add(t)
    return out
def teacher_label_safe(article, summary):
    if TEACHER_OK:
        try:
            return teacher_label(article, summary)
        except Exception as e:
            return -1, {"result": "error", "msg": str(e)}
    lab = 1 if (_nums(summary) - _nums(article)) else 0
    return lab, {"result": "Has Error" if lab else "No Error", "errors": [], "mock": True}
print("老師標籤器就緒。TEACHER_OK =", TEACHER_OK)

## 4. 執行：生摘要 → 老師標籤 → 逐行寫 JSONL

In [ ]:
# Public release: never silently produce mock training labels.
if not OLLAMA_OK or not TEACHER_OK:
    raise RuntimeError("Start Ollama and set OPENAI_API_KEY before generating training data.")
if os.path.exists(OUT_JSONL):
    raise FileExistsError("Choose a new OUT_JSONL to preserve existing data.")

rows = []
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for ai, art in enumerate(articles):
        for model in OLLAMA_MODELS:
            summ = gen_summary(art, model)
            label, rep = teacher_label_safe(art, summ)
            rec = {"id": f"a{ai}_{model}", "summarizer": model, "source": art,
                   "summary": summ, "teacher_label": label, "teacher_report": rep}
            f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush()
            rows.append({"id": rec["id"], "summarizer": model,
                         "teacher_label": label, "summary": summ[:80]})
        if (ai + 1) % 5 == 0:
            print(f"  ...{ai+1}/{len(articles)} 篇")
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print(f"完成：{len(rows)} 筆 → {OUT_JSONL} / {OUT_CSV}")
print("\n老師標籤分佈（0=一致 1=幻覺 -1=失敗）：")
print(df["teacher_label"].value_counts().to_string())
df.head(10)

## 5. 接下來（Phase 2：訓練學生）

**檢查資料品質**：理想上 0 和 1 兩類都要有足夠量（別一面倒）。若太偏，可：多放幾個生成模型、調摘要 prompt 讓它偶爾出錯、或從 CNNDM 把你既有的 flawed/正確摘要也混進來。

**規模**：概念驗證先做到幾百筆銀標籤即可（GPT-4o 成本粗估個位數美元）。

**訓練（Phase 2，需要 GPU）**：QLoRA 微調一個小學生（建議 Qwen2.5-1.5B 或 3B），學「給 source+summary → 輸出 label(+理由)」。工具：transformers + peft + trl + bitsandbytes。

**告訴我這幾項，我就幫你搭訓練 notebook**：你 GPU 的 VRAM 大約多少、最後資料集大約幾筆、想用哪個學生模型（不確定就用我建議的）。

**評估與部署**：學生 vs 老師(GPT-4o) vs qwen3:4b，在 FaithBench 金標上比準度與成本；學生再轉 GGUF + 量化 Q4_K_M 用 ollama 跑。
